In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"

orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

print(delivered_orders.shape)

(96478, 8)


In [3]:
# Connect orders to real customers
customer_orders = delivered_orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

# Connect orders to item-level revenue
order_revenue = (
    items.groupby("order_id")
    .agg(
        order_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        items_count=("order_item_id", "count"),
        unique_products=("product_id", "nunique")
    )
    .reset_index()
)

order_revenue["total_order_value"] = (
    order_revenue["order_value"] +
    order_revenue["freight_value"]
)

customer_orders = customer_orders.merge(
    order_revenue,
    on="order_id",
    how="left"
)

print(customer_orders.shape)

(96478, 14)


In [4]:
analysis_date = customer_orders["order_purchase_timestamp"].max()

customer_features = (
    customer_orders
    .groupby("customer_unique_id")
    .agg(
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        total_orders=("order_id", "nunique"),
        total_items=("items_count", "sum"),
        total_spend=("total_order_value", "sum"),
        average_order_value=("total_order_value", "mean"),
        total_freight=("freight_value", "sum"),
        unique_products=("unique_products", "sum")
    )
    .reset_index()
)

customer_features["recency_days"] = (
    analysis_date -
    customer_features["last_purchase"]
).dt.days

customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase"] -
    customer_features["first_purchase"]
).dt.days

customer_features["purchase_frequency"] = (
    customer_features["total_orders"] /
    (customer_features["customer_lifetime_days"] + 1)
)

customer_features["average_items_per_order"] = (
    customer_features["total_items"] /
    customer_features["total_orders"]
)

customer_features.head()

,customer_unique_id,first_purchase,last_purchase,total_orders,total_items,total_spend,average_order_value,total_freight,unique_products,recency_days,customer_lifetime_days,purchase_frequency,average_items_per_order
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,1,1,141.90,141.90,12.00,1,111,0,1.0,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,1,1,27.19,27.19,8.29,1,114,0,1.0,1.0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,1,1,86.22,86.22,17.22,1,536,0,1.0,1.0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,1,1,43.62,43.62,17.63,1,320,0,1.0,1.0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,1,1,196.89,196.89,16.89,1,287,0,1.0,1.0


In [5]:
customer_features = customer_features.merge(
    customers[
        [
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ].drop_duplicates("customer_unique_id"),
    on="customer_unique_id",
    how="left"
)

customer_features.head()

,customer_unique_id,first_purchase,last_purchase,total_orders,total_items,total_spend,average_order_value,total_freight,unique_products,recency_days,customer_lifetime_days,purchase_frequency,average_items_per_order,customer_city,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,1,1,141.90,141.90,12.00,1,111,0,1.0,1.0,cajamar,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,1,1,27.19,27.19,8.29,1,114,0,1.0,1.0,osasco,SP
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,1,1,86.22,86.22,17.22,1,536,0,1.0,1.0,sao jose,SC
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,1,1,43.62,43.62,17.63,1,320,0,1.0,1.0,belem,PA
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,1,1,196.89,196.89,16.89,1,287,0,1.0,1.0,sorocaba,SP


In [6]:
review_data = (
    reviews.groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

customer_reviews = (
    customer_orders[
        ["order_id", "customer_unique_id"]
    ]
    .merge(review_data, on="order_id", how="left")
    .groupby("customer_unique_id")
    .agg(
        average_review_score=("review_score", "mean"),
        number_of_reviews=("review_score", "count")
    )
    .reset_index()
)

customer_features = customer_features.merge(
    customer_reviews,
    on="customer_unique_id",
    how="left"
)

customer_features.head()

,customer_unique_id,first_purchase,last_purchase,total_orders,total_items,total_spend,average_order_value,total_freight,unique_products,recency_days,customer_lifetime_days,purchase_frequency,average_items_per_order,customer_city,customer_state,average_review_score,number_of_reviews
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,1,1,141.90,141.90,12.00,1,111,0,1.0,1.0,cajamar,SP,5.0,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,1,1,27.19,27.19,8.29,1,114,0,1.0,1.0,osasco,SP,4.0,1
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,1,1,86.22,86.22,17.22,1,536,0,1.0,1.0,sao jose,SC,3.0,1
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,1,1,43.62,43.62,17.63,1,320,0,1.0,1.0,belem,PA,4.0,1
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,1,1,196.89,196.89,16.89,1,287,0,1.0,1.0,sorocaba,SP,5.0,1


In [7]:
customer_features["R_score"] = pd.qcut(
    customer_features["recency_days"],
    5,
    labels=[5, 4, 3, 2, 1],
    duplicates="drop"
).astype(int)

customer_features["F_score"] = pd.qcut(
    customer_features["total_orders"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

customer_features["M_score"] = pd.qcut(
    customer_features["total_spend"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

customer_features["RFM_score"] = (
    customer_features["R_score"] +
    customer_features["F_score"] +
    customer_features["M_score"]
)

customer_features[[
    "customer_unique_id",
    "recency_days",
    "total_orders",
    "total_spend",
    "RFM_score"
]].head()

,customer_unique_id,recency_days,total_orders,total_spend,RFM_score
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,9
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,6
2,0000f46a3911fa3c0805444483337064,536,1,86.22,4
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,4
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,7


In [8]:
def assign_segment(score):
    if score >= 13:
        return "Champions"
    elif score >= 10:
        return "Loyal Customers"
    elif score >= 7:
        return "Potential Loyalists"
    elif score >= 5:
        return "At Risk"
    else:
        return "Low Engagement"

customer_features["customer_segment"] = (
    customer_features["RFM_score"]
    .apply(assign_segment)
)

print(
    customer_features["customer_segment"]
    .value_counts()
)

customer_segment
Potential Loyalists    38381
Loyal Customers        31509
At Risk                12273
Champions               8033
Low Engagement          3162
Name: count, dtype: int64


In [9]:
customer_features.to_csv(
    PROCESSED / "customer_features.csv",
    index=False
)

print("Customer analytical dataset created!")
print("Rows:", len(customer_features))
print("Columns:", len(customer_features.columns))

Customer analytical dataset created!
Rows: 93358
Columns: 22
